<a href="https://colab.research.google.com/github/korzhimanov/dsp-seminars/blob/main/notes/2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Практическое задание №2
## Амплитудная и фазовая модуляция: генерация, спектры, демодуляция и устойчивость к шумам

### 1. Амплитудная модуляция (АМ)

**Формула АМ-сигнала:**
$$
s_{AM}(t) = A_c \bigl[1 + a_m\cdot m(t)\bigr] \cos(2\pi f_c t)
$$
где:
- $A_c$ – амплитуда несущей,
- $f_c$ – частота несущей,
- $m(t)$ – модулирующий сигнал (сообщение), **нормированный** так, чтобы $\max |m(t)| = 1$,
- $a_m$ – глубина (амплитуда) модуляции.

**Пример генерации в Python:**
```python
fs = 10000          # частота дискретизации
t = np.linspace(0, 0.1, int(fs*0.1), endpoint=False)
fc = 1000           # несущая
fm = 100            # частота сообщения
a_m = 0.8           # амплитуда модуляции
m_t = np.sin(2*np.pi*fm*t)   # сообщение
s_am = (1 + a_m*m_t) * np.cos(2*np.pi*fc*t)
```

### 2. Фазовая модуляция (ФМ)

**Формула ФМ-сигнала:**
$$
s_{PM}(t) = A_c \cos\bigl(2\pi f_c t + \beta\, m(t)\bigr)
$$
где $\beta$ – **индекс фазовой модуляции** (максимальное отклонение фазы в радианах).

**Пример генерации:**
```python
beta = 5
s_pm = np.cos(2*np.pi*fc*t + beta * m_t)
```

### 3. Преобразование Гильберта и демодуляция

#### Аналитический сигнал
Для демодуляции АМ и ФМ сигналов будем использовать так называемый аналитический сигнал, у которого действительная часть – исходный сигнал, а мнимая – его преобразование Гильберта:
$$
\mathcal{H}[x](t) = \frac{1}{\pi} \,\text{p.v.} \int_{-\infty}^{\infty} \frac{x(\tau)}{t - \tau} \, d\tau
$$

где p.v. — главное значение интеграла по Коши.
```python
from scipy.signal import hilbert
z = hilbert(s)               # аналитический сигнал
```
Аналитический сигнал позволяет извлечь огибающую и мгновенную фазу исходного сигнала:
- **Огибающая** (envelope) – модуль аналитического сигнала: `envelope = np.abs(z)`
- **Мгновенная фаза** – аргумент: `phase = np.angle(z)`  (значения от $-\pi$ до $\pi$)

#### Демодуляция АМ
Для АМ огибающая равна $A_c[1 + m(t)]$. Чтобы восстановить $m(t)$:
```python
envelope = np.abs(hilbert(s_am))
m_recovered = envelope / Ac - 1   # если Ac=1, то просто envelope - 1
```

#### Демодуляция ФМ
Мгновенная фаза содержит линейный тренд $2\pi f_c t$ плюс полезное отклонение $\beta m(t)$. Нужно:
1. Получить мгновенную фазу и устранить скачки (unwrap).
2. Вычесть линейную составляющую.
3. Разделить на $\beta$.

```python
z = hilbert(s_pm)
phase = np.unwrap(np.angle(z))               # развёрнутая фаза
phase_no_carrier = phase - 2*np.pi*fc*t       # убираем тренд несущей
m_recovered = phase_no_carrier / beta
```
**Важно:** без `np.unwrap` скачки фазы на $\pm\pi$ приведут к ложным выбросам.

### 4. Функция `np.unwrap` – корректировка фазы

При вычислении фазы через `np.angle` результат лежит в пределах $[-\pi, \pi]$. Если истинная фаза пересекает эти границы, возникают скачки на $2\pi$. `np.unwrap` добавляет или вычитает $2\pi$ в местах скачков, восстанавливая непрерывную фазу.

```python
phase_wrapped = np.angle(z)
phase_unwrapped = np.unwrap(phase_wrapped)
```

### 5. Функция `np.std` для сравнения сигналов

Для количественной оценки ошибки между исходным сообщением $m(t)$ и восстановленным $\hat{m}(t)$ используют среднеквадратичное отклонение (СКО, Root Mean Square Error, RMSE):
$$
\text{СКО} = \sqrt{\frac{1}{N} \sum_{i=1}^{N} (x_i - y_i)^2}
$$
```python
error = np.std(m_original - m_recovered)
print(f'СКО = {error:.4f}')
```
Чем меньше значение СКО, тем точнее восстановление.

### 6. Работа с WAV-файлами

**Чтение файла:**
```python
from scipy.io import wavfile
rate, data = wavfile.read('filename.wav')
```
- `rate` – частота дискретизации (Гц)
- `data` – массив целых чисел. Форма:
  - моно: `(N,)`
  - стерео: `(N, 2)` (левый и правый каналы)
Для монофонической обработки возьмите один канал: `data = data[:, 0]` (если стерео).


**Прослушивание в Jupyter/Colab:**
```python
from IPython.display import Audio
Audio(data_float, rate=rate)
```

### 7. Амплитудный шум (генерация)

Для моделирования амплитудных помех удобно использовать **мультипликативный шум**: случайные флуктуации амплитуды несущей.

$$
y(t) = x(t) \cdot (1 + n(t))
$$
где $n(t)$ — шумовой (случайный) процесс, причём $\langle n(t)\rangle = 0 $.

**Генерация шума с помощью `np.random.randn`:**
```python
noise = np.random.randn(N)          # возвращает N случайных величины в интервале (0, 1), распределённых по Гауссу
noise_scaled = amplitude_noise * noise   # задаём уровень
s_noisy = s * (1 + noise_scaled)    # мультипликативный шум
```
Здесь `amplitude_noise` – относительный уровень шума (например, 0.2 означает ±20% флуктуации).

### 8. Дополнительные советы

- Всегда проверяйте, что модулирующий сигнал находится в диапазоне [-1, 1].
- При использовании `hilbert` надо помнить о краевых эффектах – лучше брать сигнал достаточной длины и отбрасывать края при оценке точности.
- Для построения спектров можно использовать `np.fft.rfft` (односторонний спектр)
- При сравнении сигналов и их спектров не забывайте про нормировку.
